# OpenPose Gait Parameter Analysis (from Pre-filtered JSON)

This notebook assumes the JSON file has already been post-processed (e.g. filtered with Butterworth). 
It directly loads the data using existing `src/io/keypoints_io.py` and processes the data through the exact same gait extraction pipeline as the HRNet script.

In [1]:
import os
import sys
import numpy as np
import pandas as pd

PROJECT_ROOT = "/home/projects/sipl-prj10496/project_files/projectA_repo"
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.io.keypoints_io import load_keypoints_dict_from_json
from src.domain.gait_events_detection import gait_event_detection
from src.domain.gait_feature_extraction import calculate_gait_parameters_2, calculate_spatial_parameters
import src.domain.gait_events_detection as ged

def ankle_to_pelvis_distance_dynamic(model, keypoints_dict):
    keypoints_arr = keypoints_dict['keypoints']

    if model.lower() == "body25":
        kp_arr = ged.BODY25_GAIT_KEYPOINTS
        mid_pelvis_loc = keypoints_arr[:, kp_arr["center"]["pelvis"], :2]
    else:
        raise ValueError(f"Unsupported model: {model}")
    
    left_ankle_loc = keypoints_arr[:, kp_arr["left"]["ankle"], :2]
    right_ankle_loc = keypoints_arr[:, kp_arr["right"]["ankle"], :2]

    left_ankle_to_pelvis = left_ankle_loc - mid_pelvis_loc
    right_ankle_to_pelvis = right_ankle_loc - mid_pelvis_loc

    pelvis_x = mid_pelvis_loc[:, 0]
    pelvis_velocity = np.gradient(pelvis_x)

    window = np.ones(30) / 30
    smoothed_vel = np.convolve(pelvis_velocity, window, mode='same')
    
    direction = np.zeros_like(smoothed_vel)
    direction[smoothed_vel > 0.5] = 1.0
    direction[smoothed_vel < -0.5] = -1.0
    
    dir_series = pd.Series(direction).replace(0.0, np.nan)
    dir_series = dir_series.ffill().bfill()
    direction = dir_series.fillna(1.0).to_numpy()

    left_ankle_to_pelvis[:, 0] = left_ankle_to_pelvis[:, 0] * direction
    right_ankle_to_pelvis[:, 0] = right_ankle_to_pelvis[:, 0] * direction

    return {"left_ankle_distance": left_ankle_to_pelvis,
            "right_ankle_distance": right_ankle_to_pelvis,
            "mid_pelvis_loc": mid_pelvis_loc,
            "left_ankle": left_ankle_loc,
            "right_ankle": right_ankle_loc
            }

In [2]:
input_path = "/home/projects/sipl-prj10496/project_files/data/openpose_output/20260530_133716/NL129_3_3_keypoints_20260530_133716_0_to_end.json"
fps = 60.0

print(f"Loading JSON from: {input_path}")
keypoints_dict = load_keypoints_dict_from_json(input_path, model_type="body25")
print(f"Keypoints array shape: {keypoints_dict['keypoints'].shape}")

kps = keypoints_dict['keypoints'].copy()

# 1. The Magic Fix: Convert OpenPose zeros to NaNs so pandas knows they are missing values
kps[kps == 0.0] = np.nan

# 2. Now interpolate properly
for joint_idx_id in range(kps.shape[1]):
    for dim_idx_id in range(kps.shape[2]):
        series = pd.Series(kps[:, joint_idx_id, dim_idx_id])
        # Interpolate missing values, and fill remaining edges with the nearest valid number
        interpolated = series.interpolate(limit_direction='both')
        interpolated = interpolated.bfill().ffill() 
        kps[:, joint_idx_id, dim_idx_id] = interpolated.to_numpy()

keypoints_dict['keypoints'] = kps

import src.domain.gait_events_detection as ged
from scipy.signal import find_peaks

ged.BODY25_GAIT_KEYPOINTS["center"]["pelvis"] = 8
ged.BODY25_GAIT_KEYPOINTS["left"]["ankle"] = 13
ged.BODY25_GAIT_KEYPOINTS["right"]["ankle"] = 10

distance_data = ankle_to_pelvis_distance_dynamic("body25", keypoints_dict)

# Smooth distance arrays using a Simple Moving Average (SMA) to eliminate jitter peaks
window = np.ones(15) / 15
smoothed_dist_l = np.convolve(distance_data["left_ankle_distance"][:, 0], window, mode='same')
smoothed_dist_r = np.convolve(distance_data["right_ankle_distance"][:, 0], window, mode='same')

# 3. Robust local Gait Event Detection bypassing external module 
lhs, _ = find_peaks(smoothed_dist_l, distance=40)
rhs, _ = find_peaks(smoothed_dist_r, distance=40)

lto, _ = find_peaks(-smoothed_dist_l, distance=40)
rto, _ = find_peaks(-smoothed_dist_r, distance=40)

gait_events = {
    'lhs': lhs,
    'rhs': rhs,
    'lto': lto,
    'rto': rto
}

print("\nDetected Events:")
print(f"  Left Heel Strikes (LHS): {len(gait_events['lhs'])}")
print(f"  Right Heel Strikes (RHS): {len(gait_events['rhs'])}")
print(f"  Left Toe Offs (LTO): {len(gait_events['lto'])}")
print(f"  Right Toe Offs (RTO): {len(gait_events['rto'])}")

Loading JSON from: /home/projects/sipl-prj10496/project_files/data/openpose_output/20260530_133716/NL129_3_3_keypoints_20260530_133716_0_to_end.json
Keypoints array shape: (19597, 25, 3)

Detected Events:
  Left Heel Strikes (LHS): 296
  Right Heel Strikes (RHS): 299
  Left Toe Offs (LTO): 271
  Right Toe Offs (RTO): 277


In [ ]:
# 4. Calculate Parameters
time_vector = np.array(keypoints_dict['frame_indices']) / fps

gait_params = calculate_gait_parameters_2(keypoints_dict['keypoints'], time_vector, gait_events)

# Override spatial parameters function locally to fix the hardcoded 11 and 14 for BODY25
def calculate_spatial_parameters_fixed(model, keypoints, events, scaling_factor=1.0):
    IDX_R_FOOT = 10  # Right Ankle in 15-keypoint layout
    IDX_L_FOOT = 13  # Left Ankle in 15-keypoint layout
    IDX_X = 0        # X coordinate
   
    spatial_parameters = {}
    rhs_frames = events['rhs']
    lhs_frames = events['lhs']

    if 'stepLength' not in spatial_parameters:
        spatial_parameters['stepLength'] = {}

    r_pos_at_rhs = keypoints[rhs_frames, IDX_R_FOOT, IDX_X]
    l_pos_at_rhs = keypoints[rhs_frames, IDX_L_FOOT, IDX_X]
    spatial_parameters['stepLength']['right'] = np.abs(scaling_factor * (r_pos_at_rhs - l_pos_at_rhs))

    l_pos_at_lhs = keypoints[lhs_frames, IDX_L_FOOT, IDX_X]
    r_pos_at_lhs = keypoints[lhs_frames, IDX_R_FOOT, IDX_X]
    spatial_parameters['stepLength']['left'] = np.abs(scaling_factor * (l_pos_at_lhs - r_pos_at_lhs))
    
    return spatial_parameters

spatial_params = calculate_spatial_parameters_fixed("BODY25", keypoints_dict['keypoints'], gait_events)

np.set_printoptions(precision=3, suppress=True)

print("=== Temporal Parameters ===")
for param_name, param_value in gait_params.items():
    if isinstance(param_value, dict):
        print(f"{param_name}:")
        for side, values in param_value.items():
            print(f"  {side}: {values}")
    else:
        # Print first few elements or mean if it's a large array
        if isinstance(param_value, np.ndarray) and param_value.size > 5:
            print(f"{param_name}: {param_value[:5]} ... (mean={np.nanmean(param_value):.3f})")
        else:
            print(f"{param_name}: {param_value}")

print("\n=== Spatial Parameters ===")
for param_name, param_value in spatial_params.items():
    if isinstance(param_value, dict):
        print(f"{param_name}:")
        for side, values in param_value.items():
            # Print first few elements or mean if it's a large array
            if isinstance(values, np.ndarray) and values.size > 5:
                print(f"  {side}: {values[:5]} ... (mean={np.nanmean(values):.3f})")
            else:
                print(f"  {side}: {values}")
    else:
        print(f"{param_name}: {param_value:.3f}")

In [ ]:
import matplotlib.pyplot as plt
from src.io.features_io import build_gait_step_rows_from_events_2, build_spatial_step_rows_from_events

# 1. Convert Dicts into the exact DataFrames used in video_processing_cpy.ipynb
# -------------------------------------------------------------------------------
video_name = "NL129_3_3_OpenPose"
run_hash_id = "OpenPose_JSON_Load"

step_rows = build_gait_step_rows_from_events_2(video_name, run_hash_id, gait_params)
spatial_rows = build_spatial_step_rows_from_events(video_name, run_hash_id, gait_params, spatial_params)

steps_df = pd.DataFrame(step_rows)
spatial_df = pd.DataFrame(spatial_rows)

print("✅ Successfully generated steps_df and spatial_df from OpenPose gait_params!\n")
display(steps_df.head(3))

# 2. Variability Analysis copied exactly from the previous notebook
# -------------------------------------------------------------------------------
def calculate_step_time_variability(df_steps, patient_id="Patient"):
    print(f"=== Step Time Variability Analysis for {patient_id} ===")
    
    if "valid" in df_steps.columns:
        valid_steps_df = df_steps[df_steps["valid"]].copy()
    else:
        valid_steps_df = df_steps.copy()
        
    if "step_time_s" not in valid_steps_df.columns:
        print(f"No 'step_time_s' column found.\n")
        return
    
    # Filter physiologically impossible step times (like video_processing_cpy.ipynb)
    valid_steps_df = valid_steps_df[(valid_steps_df["step_time_s"] >= 0.4) & (valid_steps_df["step_time_s"] <= 2.5)].copy()
        
    overall_mean = valid_steps_df["step_time_s"].mean()
    overall_std = valid_steps_df["step_time_s"].std()
    overall_cv = (overall_std / overall_mean) * 100 if overall_mean > 0 else 0
    
    print(f"Overall -> Mean: {overall_mean:.3f}s | Std: {overall_std:.3f}s | CV: {overall_cv:.2f}%")
    
    if "side" in valid_steps_df.columns:
        for side in valid_steps_df["side"].unique():
            side_steps_df = valid_steps_df[valid_steps_df["side"] == side]
            side_mean = side_steps_df["step_time_s"].mean()
            side_std = side_steps_df["step_time_s"].std()
            side_cv = (side_std / side_mean) * 100 if side_mean > 0 else 0
            print(f"{side.capitalize():<7} -> Mean: {side_mean:.3f}s | Std: {side_std:.3f}s | CV: {side_cv:.2f}%")
    print("\n")

calculate_step_time_variability(steps_df, video_name)

In [ ]:
# 3. Scatter Plots and Bounds exactly like the previous notebook
# -------------------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=False)

valid_steps = steps_df[steps_df["valid"]].copy()
invalid_steps = steps_df[~steps_df["valid"]].copy()

colors = {"left": "#1f77b4", "right": "#ff7f0e"}

# --- Subplot 1: Step Time ---
for side, df_side in valid_steps.groupby("side"):
    axes[0].scatter(df_side["global_step_index"], df_side["step_time_s"], label=f"{side} valid", c=colors.get(side))
for side, df_side in invalid_steps.groupby("side"):
    axes[0].scatter(df_side["global_step_index"], df_side["step_time_s"], marker="x", label=f"{side} invalid", c=colors.get(side))

axes[0].axhline(0.25, linestyle="--", color="gray", alpha=0.5)
axes[0].axhline(1.5, linestyle="--", color="gray", alpha=0.5)
axes[0].set_title("Step Time by Index")
axes[0].set_xlabel("Global Step Index")
axes[0].set_ylabel("Step Time (s)")
axes[0].legend()

# --- Subplot 2: Step Length ---
valid_spatial = spatial_df[spatial_df["valid"]].copy() if "valid" in spatial_df.columns else spatial_df.copy()
for side, df_side in valid_spatial.groupby("side"):
    axes[1].scatter(df_side["global_step_index"], df_side["step_length_px"], label=f"{side} valid", c=colors.get(side))

axes[1].set_title("Step Length by Index")
axes[1].set_xlabel("Global Step Index")
axes[1].set_ylabel("Step Length (px)")
axes[1].legend()

plt.tight_layout()
plt.show()

# 4. Distribution Histograms + IQR Outlier extraction
# -------------------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for side in ["left", "right"]:
    time_data = valid_steps.loc[valid_steps["side"] == side, "step_time_s"].dropna()
    axes[0].hist(time_data, bins=15, alpha=0.5, label=f"{side}", color=colors.get(side))
    
    spatial_data = valid_spatial.loc[valid_spatial["side"] == side, "step_length_px"].dropna()
    axes[1].hist(spatial_data, bins=15, alpha=0.5, label=f"{side}", color=colors.get(side))

axes[0].set_title("Step Time Distribution")
axes[0].set_xlabel("Step Time (s)")
axes[0].set_ylabel("Count")
axes[0].legend()

axes[1].set_title("Step Length Distribution")
axes[1].set_xlabel("Step Length (px)")
axes[1].set_ylabel("Count")
axes[1].legend()

plt.tight_layout()
plt.show()

def iqr_bounds(x, k=1.5):
    q1 = x.quantile(0.25)
    q3 = x.quantile(0.75)
    iqr = q3 - q1
    return max(q1 - k * iqr, 0), q3 + k * iqr

spatial_bounds = {}
for side, df_side in valid_spatial.groupby("side"):
    spatial_bounds[side] = iqr_bounds(df_side["step_length_px"])

print("Step Length Outlier Bounds (IQR):")
print(spatial_bounds)